# Aether 2 Full-Stack Colab Prototype

This notebook runs the current Aether 2 trainer with every implemented mechanism active at once: CSSC, GGR, Rosetta, baseline, and Fluid Power Allocation.

Recommended first run: keep the architecture intact, use a small real JSONL subset, and keep the step count low.

In [13]:
# Install runtime dependencies for Colab.
%pip install -q torch torchvision torchaudio safetensors numpy tqdm rich datasets huggingface_hub bitsandbytes

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
import getpass
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/xcrrr/aether.git'
REPO_DIR = Path('/content/aether')

if not REPO_DIR.exists():
    # First try anonymous clone (works for public repos).
    anon = subprocess.run(
        ['git', 'clone', REPO_URL, str(REPO_DIR)],
        capture_output=True,
        text=True,
    )
    if anon.returncode != 0:
        print('Anonymous clone failed. Repo is likely private.')
        token = getpass.getpass('Paste GitHub PAT with repo read access: ')
        auth_url = REPO_URL.replace('https://', f'https://{token}@')
        subprocess.run(['git', 'clone', auth_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Working directory:', REPO_DIR)

Working directory: /content/aether


In [16]:
# Resolve dataset source automatically and build subset without external helper scripts.
from pathlib import Path
import json
import random

SUBSET_PATH = REPO_DIR / 'data' / 'aether_train_colab.jsonl'
CHECKPOINT_DIR = Path('/content/drive/MyDrive/aether/checkpoints_colab_full')
LOG_FILE = Path('/content/drive/MyDrive/aether/logs/aether_colab_full.log')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    REPO_DIR / 'data' / 'aether_train.jsonl',
    Path('/content/drive/MyDrive/aether/data/aether_train.jsonl'),
    Path('/content/drive/MyDrive/aether_train.jsonl'),
    Path('/content/drive/MyDrive/data/aether_train.jsonl'),
]

SOURCE_DATA = next((p for p in candidates if p.exists()), None)
if SOURCE_DATA is None:
    print('Could not find aether_train.jsonl in expected locations.')
    print('Creating bootstrap dataset in repo so you can start training now...')
    BOOTSTRAP_PATH = REPO_DIR / 'data' / 'aether_train_bootstrap.jsonl'
    BOOTSTRAP_PATH.parent.mkdir(parents=True, exist_ok=True)

    seq_len = 512
    n_records = 8192
    vocab_low = 10
    vocab_high = 32000
    rng = random.Random(42)

    with BOOTSTRAP_PATH.open('w', encoding='utf-8') as f:
        for _ in range(n_records):
            input_ids = [rng.randint(vocab_low, vocab_high) for _ in range(seq_len)]
            labels = input_ids[1:] + [-100]
            record = {
                'input_ids': input_ids,
                'labels': labels,
                'trust_score': 90.0,
                'source': 'bootstrap',
            }
            f.write(json.dumps(record, ensure_ascii=True) + '\n')

    SOURCE_DATA = BOOTSTRAP_PATH
    print(f'Bootstrap dataset created: {SOURCE_DATA} ({n_records} records)')

print(f'Using source dataset: {SOURCE_DATA}')
print(f'Writing subset to: {SUBSET_PATH}')

# Build subset directly in-notebook (no prepare_colab_subset.py dependency).
def _is_valid(rec: dict) -> bool:
    required = {'input_ids', 'labels', 'trust_score', 'source'}
    if not required.issubset(rec.keys()):
        return False
    if not isinstance(rec['input_ids'], list) or not isinstance(rec['labels'], list):
        return False
    if len(rec['input_ids']) == 0 or len(rec['input_ids']) != len(rec['labels']):
        return False
    return True

kept = 0
max_records = 4096
SUBSET_PATH.parent.mkdir(parents=True, exist_ok=True)
with SOURCE_DATA.open('r', encoding='utf-8') as src, SUBSET_PATH.open('w', encoding='utf-8') as dst:
    for line in src:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if not _is_valid(rec):
            continue
        dst.write(json.dumps(rec, ensure_ascii=True) + '\n')
        kept += 1
        if kept >= max_records:
            break

if kept == 0:
    raise RuntimeError('Failed to build subset: no valid records found.')

print(f'Subset ready: {SUBSET_PATH} ({kept} records)')

Could not find aether_train.jsonl in expected locations.
Creating bootstrap dataset in repo so you can start training now...
Bootstrap dataset created: /content/aether/data/aether_train_bootstrap.jsonl (8192 records)
Using source dataset: /content/aether/data/aether_train_bootstrap.jsonl
Writing subset to: /content/aether/data/aether_train_colab.jsonl
Subset ready: /content/aether/data/aether_train_colab.jsonl (4096 records)


In [17]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())

CUDA available: True
GPU: Tesla T4
BF16 supported: True


In [18]:
import json
import random
import shutil
import subprocess
from pathlib import Path
import torch

# --- Run profile selector ---
# 'final' = README-style long run, 'poc' = quick architecture signal run.
RUN_PROFILE = 'final'

if RUN_PROFILE == 'final':
    TARGET_MAX_STEPS = 50_000
    TARGET_LOG_EVERY = 20
    TARGET_CHECKPOINT_EVERY = 2_000
    TARGET_MICRO_BATCH = 1
    TARGET_GRAD_ACCUM = 1
elif RUN_PROFILE == 'poc':
    TARGET_MAX_STEPS = 2_000
    TARGET_LOG_EVERY = 20
    TARGET_CHECKPOINT_EVERY = 500
    TARGET_MICRO_BATCH = 1
    TARGET_GRAD_ACCUM = 1
else:
    raise ValueError("RUN_PROFILE must be 'final' or 'poc'")

print(f'Run profile: {RUN_PROFILE}')
print(f'Steps={TARGET_MAX_STEPS}, log_every={TARGET_LOG_EVERY}, checkpoint_every={TARGET_CHECKPOINT_EVERY}')

# --- Self-healing setup: rebuild required paths/files if runtime was reset ---
REPO_DIR = Path(globals().get('REPO_DIR', '/content/aether'))
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f'Repo directory not found: {REPO_DIR}. Run Cell 4 first to clone/open repo.'
    )

SUBSET_PATH = Path(globals().get('SUBSET_PATH', REPO_DIR / 'data' / 'aether_train_colab.jsonl'))
CHECKPOINT_DIR = Path(globals().get('CHECKPOINT_DIR', '/content/drive/MyDrive/aether/checkpoints_colab_full'))
LOG_FILE = Path(globals().get('LOG_FILE', '/content/drive/MyDrive/aether/logs/aether_colab_full.log'))
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

if not SUBSET_PATH.exists():
    raise FileNotFoundError(
        f'Subset missing: {SUBSET_PATH}. Run Cell 5 first to create subset dataset.'
    )

subset_size_mb = SUBSET_PATH.stat().st_size / (1024 * 1024)
print(f'Subset file: {SUBSET_PATH} ({subset_size_mb:.2f} MB)')

with SUBSET_PATH.open('r', encoding='utf-8') as f:
    first_line = f.readline().strip()
    if not first_line:
        raise RuntimeError('Subset file is empty.')
    first = json.loads(first_line)

required = {'input_ids', 'labels', 'trust_score', 'source'}
missing = sorted(required.difference(first.keys()))
if missing:
    raise RuntimeError(f'Subset schema invalid, missing keys: {missing}')
if len(first['input_ids']) != len(first['labels']):
    raise RuntimeError('Subset schema invalid: input_ids and labels length mismatch.')

print(
    f"First record looks valid: len(input_ids)={len(first['input_ids'])}, "
    f"len(labels)={len(first['labels'])}, source={first['source']}"
 )

help_out = subprocess.run(
    ['python', 'aether2_train.py', '--help'],
    capture_output=True,
    text=True,
    check=False,
    cwd=str(REPO_DIR),
).stdout

supports_checkpoint_dir = '--checkpoint-dir' in help_out
supports_log_file = '--log-file' in help_out
supports_no_bf16 = '--no-bf16' in help_out
supports_log_every = '--log-every' in help_out

ACTIVE_CHECKPOINT_DIR = CHECKPOINT_DIR if supports_checkpoint_dir else (REPO_DIR / 'checkpoints_aether2')
ACTIVE_LOG_FILE = LOG_FILE if supports_log_file else (REPO_DIR / 'logs' / 'aether_build.log')

cmd = [
    'python', 'aether2_train.py',
    '--fpa',
    '--max-steps', str(TARGET_MAX_STEPS),
    '--micro-batch', str(TARGET_MICRO_BATCH),
    '--grad-accum', str(TARGET_GRAD_ACCUM),
    '--checkpoint-every', str(TARGET_CHECKPOINT_EVERY),
    '--data-path', str(SUBSET_PATH),
]

if supports_log_every:
    cmd.extend(['--log-every', str(TARGET_LOG_EVERY)])
if supports_checkpoint_dir:
    cmd.extend(['--checkpoint-dir', str(CHECKPOINT_DIR)])
if supports_log_file:
    cmd.extend(['--log-file', str(LOG_FILE)])
if (not torch.cuda.is_bf16_supported()) and supports_no_bf16:
    cmd.append('--no-bf16')

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False, cwd=str(REPO_DIR))
print('Exit code:', result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        'Training command failed. Check the log file in Cell 8 for root cause. '
        'If OOM, keep all mechanisms on and lower micro-batch/grad-accum.'
    )

if not supports_log_file and Path(ACTIVE_LOG_FILE).exists():
    LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ACTIVE_LOG_FILE, LOG_FILE)
    print(f'Copied log to Drive: {LOG_FILE}')

if not supports_checkpoint_dir and Path(ACTIVE_CHECKPOINT_DIR).exists():
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    for item in Path(ACTIVE_CHECKPOINT_DIR).iterdir():
        dst = CHECKPOINT_DIR / item.name
        if item.is_dir() and not dst.exists():
            shutil.copytree(item, dst)
    print(f'Synced checkpoints to Drive: {CHECKPOINT_DIR}')

Run profile: final
Steps=50000, log_every=20, checkpoint_every=2000
Subset file: /content/aether/data/aether_train_colab.jsonl (26.89 MB)
First record looks valid: len(input_ids)=512, len(labels)=512, source=bootstrap
Running: python aether2_train.py --fpa --max-steps 50000 --micro-batch 1 --grad-accum 1 --checkpoint-every 2000 --data-path /content/aether/data/aether_train_colab.jsonl


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt

log_path = Path(ACTIVE_LOG_FILE) if 'ACTIVE_LOG_FILE' in globals() else Path(LOG_FILE)
ckpt_path = Path(ACTIVE_CHECKPOINT_DIR) if 'ACTIVE_CHECKPOINT_DIR' in globals() else Path(CHECKPOINT_DIR)

print(f'Inspecting log: {log_path}')
print(f'Inspecting checkpoints: {ckpt_path}')

if log_path.exists():
    !tail -n 80 {log_path}
else:
    print('Log file not found yet.')

if ckpt_path.exists():
    !ls -lah {ckpt_path}
else:
    print('Checkpoint directory not found yet.')

if log_path.exists():
    pattern = re.compile(
        r'Step\s+(\d+)\s+\|\s+LR=([0-9.eE+-]+)\s+\|\s+CE=([0-9.eE+-]+)\s+'
        r'\|\s+PPL=([0-9.eE+-]+)\s+\|\s+GGR-aux=([0-9.eE+-]+)\s+'
        r'\|\s+VRAM=([0-9.eE+-]+)/([0-9.eE+-]+)GiB\s+\|\s+([0-9.eE+-]+)tok/s\s+'
        r'\|\s+Baseline Δ=([+\-0-9.eE]+)%'
    )
    steps, ce, ppl, ggr_aux = [], [], [], []
    vram_used, tok_s, delta = [], [], []

    with log_path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            m = pattern.search(line)
            if not m:
                continue
            steps.append(int(m.group(1)))
            ce.append(float(m.group(3)))
            ppl.append(float(m.group(4)))
            ggr_aux.append(float(m.group(5)))
            vram_used.append(float(m.group(6)))
            tok_s.append(float(m.group(8)))
            delta.append(float(m.group(9)))

    if len(steps) == 0:
        print('No step metrics found yet. Let training run until at least one log interval.')
    else:
        print(f'Parsed {len(steps)} metric points from log.')
        interval = (steps[1] - steps[0]) if len(steps) > 1 else None
        print(f'Logging interval: {interval if interval is not None else "N/A"} steps')

        # PoC summary in percentages for presentation.
        ppl_drop_pct = ((ppl[0] - ppl[-1]) / ppl[0] * 100.0) if ppl[0] != 0 else 0.0
        ce_drop_pct = ((ce[0] - ce[-1]) / ce[0] * 100.0) if ce[0] != 0 else 0.0
        tok_s_change_pct = ((tok_s[-1] - tok_s[0]) / tok_s[0] * 100.0) if tok_s[0] != 0 else 0.0
        baseline_delta_mean = sum(delta) / len(delta)
        ggr_aux_change_pct = ((ggr_aux[0] - ggr_aux[-1]) / ggr_aux[0] * 100.0) if ggr_aux[0] != 0 else 0.0

        print('\nPoC % Summary')
        print(f'- PPL change: {ppl[0]:.4f} -> {ppl[-1]:.4f} ({ppl_drop_pct:+.2f}%)')
        print(f'- CE change: {ce[0]:.4f} -> {ce[-1]:.4f} ({ce_drop_pct:+.2f}%)')
        print(f'- Throughput change: {tok_s[0]:.2f} -> {tok_s[-1]:.2f} tok/s ({tok_s_change_pct:+.2f}%)')
        print(f'- Mean Baseline Δ: {baseline_delta_mean:+.2f}%')
        print(f'- GGR aux change: {ggr_aux[0]:.4f} -> {ggr_aux[-1]:.4f} ({ggr_aux_change_pct:+.2f}%)')

        fig, axes = plt.subplots(2, 2, figsize=(14, 9))

        axes[0, 0].plot(steps, ppl, label='Perplexity (PPL)')
        axes[0, 0].set_title('Perplexity vs Step')
        axes[0, 0].set_xlabel('Step')
        axes[0, 0].set_ylabel('PPL')
        axes[0, 0].grid(True, alpha=0.3)

        axes[0, 1].plot(steps, ce, label='Cross-Entropy', color='tab:orange')
        axes[0, 1].set_title('Cross-Entropy vs Step')
        axes[0, 1].set_xlabel('Step')
        axes[0, 1].set_ylabel('CE')
        axes[0, 1].grid(True, alpha=0.3)

        axes[1, 0].plot(steps, tok_s, label='Tokens/s', color='tab:green')
        axes[1, 0].set_title('Throughput vs Step')
        axes[1, 0].set_xlabel('Step')
        axes[1, 0].set_ylabel('tok/s')
        axes[1, 0].grid(True, alpha=0.3)

        axes[1, 1].plot(steps, vram_used, label='VRAM used', color='tab:red')
        axes[1, 1].set_title('VRAM Used vs Step')
        axes[1, 1].set_xlabel('Step')
        axes[1, 1].set_ylabel('GiB')
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        fig2, ax2 = plt.subplots(1, 1, figsize=(10, 4))
        ax2.plot(steps, delta, label='Baseline Δ%')
        ax2.plot(steps, ggr_aux, label='GGR aux')
        ax2.set_title('Additional Signals vs Step')
        ax2.set_xlabel('Step')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        plt.tight_layout()
        plt.show()